In [7]:
import pandas as pd
import editdistance
import sys
sys.path.insert(0, '..')
from src.normalisation import normalize

# Charger 200 lignes du val set
val_df = pd.read_csv('../dataset_nlp/splits/val.csv').sample(200, random_state=42)
print(f"Échantillon : {len(val_df)} lignes")
print(f"Colonnes : {val_df.columns.tolist()}")
print(val_df['text'].iloc[0])

Échantillon : 200 lignes
Colonnes : ['text', 'image_path', 'language', 'century', 'shelfmark', 'project', 'source_corpus']
Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion


In [8]:
import numpy as np

def cer(pred: str, gt: str) -> float:
    if len(gt) == 0:
        return 0.0
    return editdistance.eval(pred, gt) / len(gt)

results = []
for _, row in val_df.iterrows():
    original = str(row['text'])
    normalized = normalize(original)
    c = cer(normalized, original)
    results.append({
        'original': original,
        'normalized': normalized,
        'cer_modification': c,
        'changed': original != normalized
    })

df_results = pd.DataFrame(results)
n_changed = df_results['changed'].sum()
cer_mean = df_results['cer_modification'].mean()

print(f"Lignes modifiées : {n_changed}/200 ({n_changed/2:.1f}%)")
print(f"Taux de modification moyen (CER) : {cer_mean:.4f} ({cer_mean*100:.2f}%)")
print(f"\nExemples de modifications :")
changed = df_results[df_results['changed']][['original','normalized']].head(5)
for _, r in changed.iterrows():
    print(f"  AVANT : {r['original']}")
    print(f"  APRÈS : {r['normalized']}")
    print()

Lignes modifiées : 128/200 (64.0%)
Taux de modification moyen (CER) : 0.0261 (2.61%)

Exemples de modifications :
  AVANT : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion
  APRÈS : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion

  AVANT : alèrent où ce faisoit ledit debat, et y alant icelluy suppliant rencontra ung nommé Anthoine de France auquel il demanda
  APRÈS : alèrent où ce faisoit ledit debat, et y alant icelluy suppliant rencontra ung nommé Anthoine de France auquel il demanda

  AVANT : Loys, par la grace de Dieu roy de France. Savoir faisons à tous, presens et avenir, nous avoir receue l’umble supplicacion
  APRÈS : Loys, par la grace de Dieu roy de France. Savoir faisons à tous, presens et avenir, nous avoir receue l’umble supplicacion

  AVANT : Et quant vous m’eussiez demandé du poisson, je vous en eusse bien donné,

In [9]:
from src.normalisation import normalize_uv, normalize_ij, expand_abbreviations, normalize_unicode

# Debug sur les cas problématiques
tests = ["je", "Jehan", "desrober"]
for t in tests:
    a1 = expand_abbreviations(t)
    a2 = normalize_unicode(a1)
    a3 = normalize_uv(a2)
    a4 = normalize_ij(a3)
    print(f"{t!r} → abbr={a1!r} → nfd={a2!r} → uv={a3!r} → ij={a4!r}")

'je' → abbr='je' → nfd='je' → uv='je' → ij='ie'
'Jehan' → abbr='Jehan' → nfd='Jehan' → uv='Jehan' → ij='iehan'
'desrober' → abbr='desireober' → nfd='desireober' → uv='desireober' → ij='desireober'


In [10]:
import importlib
import src.normalisation
importlib.reload(src.normalisation)
from src.normalisation import normalize, normalize_uv, normalize_ij, expand_abbreviations, normalize_unicode

# Vérification rapide
print(expand_abbreviations("desrober"))  # doit rester 'desrober'
print(normalize_ij("je vous"))           # doit rester 'je vous'
print(normalize_ij("jour"))              # doit rester 'jour' (j devant o... attends)

desrober
je vous
iour


In [11]:
results = []
for _, row in val_df.iterrows():
    original = str(row['text'])
    normalized = normalize(original)
    c = cer(normalized, original)
    results.append({
        'original': original,
        'normalized': normalized,
        'cer_modification': c,
        'changed': original != normalized
    })

df_results = pd.DataFrame(results)
n_changed = df_results['changed'].sum()
cer_mean = df_results['cer_modification'].mean()

print(f"Lignes modifiées : {n_changed}/200 ({n_changed/2:.1f}%)")
print(f"Taux de modification moyen (CER) : {cer_mean:.4f} ({cer_mean*100:.2f}%)")
print(f"\nExemples de modifications :")
changed = df_results[df_results['changed']][['original','normalized']].head(5)
for _, r in changed.iterrows():
    print(f"  AVANT : {r['original']}")
    print(f"  APRÈS : {r['normalized']}")
    print()

Lignes modifiées : 122/200 (61.0%)
Taux de modification moyen (CER) : 0.0251 (2.51%)

Exemples de modifications :
  AVANT : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion
  APRÈS : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion

  AVANT : alèrent où ce faisoit ledit debat, et y alant icelluy suppliant rencontra ung nommé Anthoine de France auquel il demanda
  APRÈS : alèrent où ce faisoit ledit debat, et y alant icelluy suppliant rencontra ung nommé Anthoine de France auquel il demanda

  AVANT : Loys, par la grace de Dieu roy de France. Savoir faisons à tous, presens et avenir, nous avoir receue l’umble supplicacion
  APRÈS : Loys, par la grace de Dieu roy de France. Savoir faisons à tous, presens et avenir, nous avoir receue l’umble supplicacion

  AVANT : Et quant vous m’eussiez demandé du poisson, je vous en eusse bien donné,

In [ ]:
import json
from datetime import datetime

entry = {
    "step": "nlp_normalisation_regles",
    "date": "2026-06-18",
    "description": (
        "Normalisation orthographique par règles sur 200 lignes val set. "
        "Pipeline: abréviations → NFD → u/v → i/j → ponctuation."
    ),
    "n_lignes": 200,
    "lignes_modifiees": int(n_changed),
    "pct_modifiees": round(n_changed/2, 1),
    "cer_modification_moyen": round(cer_mean, 4),
    "seuil_validation": 0.10,
    "statut": "VALIDE (CER < 10%)",
    "notes": (
        "Règles appliquées: Unicode NFD, abréviations médiévales (ñ→nn, ꝵ→rum), "
        "u initial devant voyelle → v, j initial devant a/o/u → i. "
        "Faux positifs corrigés: sr→sire supprimé, j devant e/i conservé."
    )
}

with open("../experiments/journal.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("✅ Entrée journal ajoutée")